In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
import os
import subprocess
import warnings
warnings.simplefilter('ignore')
import sys
import time

def cache_model_background(path, num_workers=16):
    """Start cache_model in background using safetensors, return immediately."""

    script = f'''
import os, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from safetensors.torch import load_file

path = "{path}"
exts = (".bin", ".pt", ".safetensors")
num_workers = {num_workers}

def warmup_file(fpath):
    if fpath.endswith(".safetensors"):
        data = load_file(fpath, device="cpu")
        # return fpath, sum(t.nbytes for t in data.values())

    # fallback for .bin/.pt
    total = 0
    with open(fpath, "rb") as f:
        while chunk := f.read(256 * 1024 * 1024):
            total += len(chunk)
    return fpath, total

if os.path.isdir(path):
    files = [
        os.path.join(root, name)
        for root, _, names in os.walk(path)
        for name in names
        if name.endswith(exts)
    ]
    files.sort()
else:
    files = [path]

print(f"[cache_model] {{len(files)}} file(s), {{num_workers}} worker(s)")
t0 = time.time()
total_bytes = 0

with ThreadPoolExecutor(max_workers=num_workers) as pool:
    futures = {{pool.submit(warmup_file, f): f for f in files}}
    for i, fut in enumerate(as_completed(futures), 1):
        fpath, n = fut.result()
        total_bytes += n
        print(f"[{{i}}/{{len(files)}}] cached {{os.path.basename(fpath)}}")

elapsed = time.time() - t0
gb = total_bytes / 1024**3
print(f"[cache_model] total ≈ {{gb:.2f}} GB in {{elapsed:.2f}}s")
'''

    proc = subprocess.Popen(
        ["python", "-c", script],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True
    )
    print(f"[cache_model] started in background, pid={proc.pid}")
    return proc

# Usage
proc = cache_model_background("/kaggle/input/models/nguyennguyen599/qwen3-4b-grids15-sft139-ckpt-v1/transformers/step_4000/1")
proc = cache_model_background("/kaggle/input/datasets/nguyennguyen599/qwen-38-27b-rl-32-nvfp4")
proc = cache_model_background("/kaggle/input/models/nguyennguyen599/qwen3-8/transformers/unsloth-qwen3.8-27b-nvfp4/1")
proc = cache_model_background("/kaggle/input/models/nguyennguyen599/qwen3-8/transformers/z-lab-qwen3.8-27b-dflash2/1")
proc = cache_model_background("/kaggle/input/datasets/nguyennguyen599/ista-daslab-qwen3-8-flash-next-gsq-rco-gguf")

In [ ]:
import subprocess

proc = subprocess.Popen(
    """
    sudo rm -rf /usr/local/cuda-12.8 ;
    sudo dpkg -i /kaggle/input/datasets/nguyennguyen599/cu13-install/cuda-keyring_1.1-1_all.deb &&
    cd /kaggle/input/datasets/nguyennguyen599/cu13-install &&
    sudo dpkg -i ./*.deb
    """,
    shell=True,
)

print("CUDA installation started.")

In [ ]:
!rm -r /kaggle/working/*

In [ ]:
!cp /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/* /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/patches /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/third_party /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/pipeline_runtime_bootstrap /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/vllm_delta_mhar_plugin/vllm_delta_mhar_plugin /kaggle/working/

In [ ]:
%env UV_OFFLINE=1

from concurrent.futures import ThreadPoolExecutor
import glob
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
import subprocess

# %env UV_FIND_LINKS=/kaggle/input/datasets/nguyennguyen599/vllm-wheels/wheels/wheels
%env UV_FIND_LINKS=/kaggle/input/datasets/nguyennguyen599/vllm-wheels/wheels/wheels
%env UV_OFFLINE=1
WHEELS = Path(
    # "/kaggle/input/datasets/nguyennguyen599/vllm-wheels/wheels/wheels"
    "/kaggle/input/datasets/nguyennguyen599/vllm-wheels/wheels/wheels"
)
WORKING = Path("/kaggle/working")

os.environ["UV_FIND_LINKS"] = str(WHEELS)
os.environ["UV_OFFLINE"] = "1"
WHEELS = Path(
    "/dev/shm/wheels"
    # "/kaggle/input/vllm-wheels/wheels/wheels"
)

src_dir = os.environ["UV_FIND_LINKS"]
dst_dir = "/dev/shm/wheels"
os.makedirs(dst_dir, exist_ok=True)

items_to_copy = os.listdir(src_dir)


def copy_item(item_name):
    src_path = os.path.join(src_dir, item_name)
    dst_path = os.path.join(dst_dir, item_name)
    try:
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            return f"Done copy folder: {item_name}"
        else:
            shutil.copy2(src_path, dst_path)
            return f"Done copy file: {item_name}"
    except Exception as e:
        return f"Error copy {item_name}: {e}"

print(f"Start copy {len(items_to_copy)} folder from {src_dir} to {dst_dir}...")

with ThreadPoolExecutor() as executor:
    results = executor.map(copy_item, items_to_copy)

for result in results:
    print(result)
    
print("done all")
os.environ["UV_OFFLINE"] = "1"

os.environ["UV_FIND_LINKS"] = "/dev/shm/wheels"
%env UV_FIND_LINKS=/dev/shm/wheels

In [ ]:
import os
import glob
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
import subprocess

# Supports both exact paths and wildcard patterns.
copy_patterns = [
    "/kaggle/input/datasets/nguyennguyen599/unsloth-main/unsloth-zoo",
    "/kaggle/input/datasets/nguyennguyen599/unsloth-main/unsloth",
    "/kaggle/input/datasets/nguyennguyen599/unsloth-main/peft",
    str(WHEELS / "liger_kernel*/liger_kernel"),
    str(WHEELS / "quack-kernels*/quack-kernels")
]

sources = []
for pattern in copy_patterns:
    matches = glob.glob(pattern)

    if not matches:
        print(f"WARNING: no match: {pattern}")
    else:
        sources.extend(map(Path, matches))

def copy_item(src: Path):
    dst = WORKING / src.name

    if src.is_dir():
        shutil.copytree(
            src,
            dst,
            dirs_exist_ok=True,
            copy_function=shutil.copyfile,  # Faster: does not copy metadata
        )
    else:
        shutil.copyfile(src, dst)

    return src, dst


# A small worker count is usually fastest on Kaggle storage.
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(copy_item, src) for src in sources]

    for future in as_completed(futures):
        try:
            src, dst = future.result()
            print(f"Copied: {src} -> {dst}")
        except Exception as error:
            print(f"Copy failed: {error}")

!uv pip install --no-build-isolation vllm torch torchvision -U
!uv pip install torchaudio torchcodec -U
!uv pip install mamba-ssm==2.3.2.post1 --no-deps
!uv pip install --no-build-isolation /kaggle/working/unsloth-zoo /kaggle/working/unsloth /kaggle/working/peft
!uv pip install flash_attn causal_conv1d bitsandbytes xformers
!uv pip install ${UV_FIND_LINKS}/transformer_engine-*
!uv pip install apex
!uv pip install redis
!uv pip install --no-build-isolation /kaggle/working/liger_kernel
!uv pip install --no-build-isolation \
    flashinfer-python flashinfer-cubin flashinfer-jit-cache
    # flashinfer-python==0.6.15.post1 flashinfer-cubin==0.6.15.post1 flashinfer-jit-cache==0.6.15.post1

!uv pip install --no-build-isolation /kaggle/working/quack-kernels

!uv pip install transformers==5.20.0

!uv pip install --no-build-isolation --no-deps \
    nvidia-cudnn-cu13==9.24.0.43 nvidia-nvshmem-cu13==3.6.5 nvidia-nccl-cu13==2.30.7
    # nvidia-cudnn-cu13==9.20.0.48 nvidia-nvshmem-cu13==3.6.5 nvidia-nccl-cu13==2.30.7
    # nvidia-cudnn-cu13==9.20.0.48 nvidia-nvshmem-cu13==3.6.5 vidia-nccl-cu13==2.29.7

# !uv pip install nvidia-cutlass-dsl==4.8.0.dev0
# !uv pip install nvidia-cutlass-dsl~=4.7.1 nvidia-cutlass-dsl-libs-cu13==4.7.1
# !uv pip install nvidia-cutlass-dsl~=4.8.0 nvidia-cutlass-dsl-libs-cu13==4.8.0
!uv pip install nvidia-cutlass-dsl~=4.8.1 nvidia-cutlass-dsl-libs-cu13==4.8.1


proc.wait()

if proc.returncode != 0:
    raise RuntimeError("CUDA installation failed")

print("CUDA installation finished.")

In [ ]:
!uv pip uninstall tensorflow jax

In [ ]:
!ps aux | grep vllm

In [ ]:
!rm -r /dev/shm/wheels

In [ ]:
%%writefile /usr/local/lib/python3.13/dist-packages/vllm/model_executor/models/qwen3_dflash2.py
# SPDX-License-Identifier: Apache-2.0
# SPDX-FileCopyrightText: Copyright contributors to the vLLM project

import torch
import torch.nn.functional as F
from torch import nn

from vllm import _custom_ops as ops
from vllm.compilation.backends import set_model_tag
from vllm.compilation.decorators import support_torch_compile
from vllm.config import CacheConfig, VllmConfig
from vllm.distributed import get_tp_group
from vllm.logger import init_logger
from vllm.model_executor.layers.linear import ReplicatedLinear
from vllm.model_executor.layers.logits_processor import LogitsProcessor
from vllm.model_executor.layers.quantization.base_config import QuantizationConfig
from vllm.platforms import current_platform
from vllm.triton_utils import tl, triton
from vllm.utils.torch_utils import (
    direct_register_custom_op,
    get_accelerator_view_from_cpu_tensor,
)

from .qwen3_dflash import (
    DFlashQwen3DecoderLayer,
    DFlashQwen3ForCausalLM,
    DFlashQwen3Model,
)
from .utils import maybe_prefix

logger = init_logger(__name__)


@triton.jit
def _dflash2_grouped_conv_kernel(
    x_ptr,
    delta_ptr,
    base_ptr,
    output_ptr,
    x_stride_row,
    delta_stride_row,
    delta_stride_tap,
    base_stride_tap,
    output_stride_row,
    NUM_CHANNELS: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    GROUP_SIZE: tl.constexpr,
    TAPS: tl.constexpr,
    ELEMENT_BLOCK: tl.constexpr,
) -> None:
    row = tl.program_id(0) // triton.cdiv(NUM_CHANNELS, ELEMENT_BLOCK)
    col_block = tl.program_id(0) % triton.cdiv(NUM_CHANNELS, ELEMENT_BLOCK)
    channels = col_block * ELEMENT_BLOCK + tl.arange(0, ELEMENT_BLOCK)
    mask = channels < NUM_CHANNELS
    groups = channels // GROUP_SIZE
    position = row % BLOCK_SIZE

    delta_row = delta_ptr + row * delta_stride_row
    x_row = x_ptr + row * x_stride_row
    accumulator = (
        tl.load(base_ptr + channels, mask=mask, other=0.0).to(tl.float32)
        + tl.load(delta_row + groups, mask=mask, other=0.0).to(tl.float32)
    ) * tl.load(x_row + channels, mask=mask, other=0.0).to(tl.float32)

    for tap in tl.static_range(1, TAPS):
        tap_mask = mask & (position >= tap)
        coefficient = tl.load(
            base_ptr + tap * base_stride_tap + channels,
            mask=tap_mask,
            other=0.0,
        ).to(tl.float32) + tl.load(
            delta_row + tap * delta_stride_tap + groups,
            mask=tap_mask,
            other=0.0,
        ).to(tl.float32)
        x = tl.load(
            x_ptr + (row - tap) * x_stride_row + channels,
            mask=tap_mask,
            other=0.0,
        ).to(tl.float32)
        accumulator += coefficient * x

    tl.store(
        output_ptr + row * output_stride_row + channels,
        accumulator,
        mask=mask,
    )


def dflash2_grouped_conv_impl(
    x: torch.Tensor,
    delta: torch.Tensor,
    base: torch.Tensor,
    block_size: int,
    group_size: int,
) -> torch.Tensor:
    """DFlash2 grouped convolution adapted from LightSeek TokenSpeed PR #1399."""
    num_rows, num_channels = x.shape
    output = torch.empty_like(x)
    if num_rows == 0:
        return output

    element_block = 1024 if num_rows >= 128 and num_channels % 1024 == 0 else 512
    grid = (num_rows * triton.cdiv(num_channels, element_block),)
    _dflash2_grouped_conv_kernel[grid](
        x,
        delta,
        base,
        output,
        x.stride(0),
        delta.stride(0),
        delta.stride(1),
        base.stride(0),
        output.stride(0),
        NUM_CHANNELS=num_channels,
        BLOCK_SIZE=block_size,
        GROUP_SIZE=group_size,
        TAPS=base.shape[0],
        ELEMENT_BLOCK=element_block,
        num_warps=4,
    )
    return output


def dflash2_grouped_conv_fake(
    x: torch.Tensor,
    delta: torch.Tensor,
    base: torch.Tensor,
    block_size: int,
    group_size: int,
) -> torch.Tensor:
    return torch.empty_like(x)


direct_register_custom_op(
    op_name="dflash2_grouped_conv",
    op_func=dflash2_grouped_conv_impl,
    fake_impl=dflash2_grouped_conv_fake,
    tags=(torch.Tag.needs_fixed_stride_order,),
)


def dflash2_grouped_conv(
    x: torch.Tensor,
    delta: torch.Tensor,
    base: torch.Tensor,
    block_size: int,
    group_size: int,
) -> torch.Tensor:
    return torch.ops.vllm.dflash2_grouped_conv(x, delta, base, block_size, group_size)


def _grouped_conv(
    hidden_states: torch.Tensor,
    delta: torch.Tensor,
    base: torch.Tensor,
    block_size: int,
    num_groups: int,
    group_size: int,
    taps: int,
) -> torch.Tensor:
    if hidden_states.is_cuda:
        return dflash2_grouped_conv(hidden_states, delta, base, block_size, group_size)

    blocks = hidden_states.unflatten(-1, (num_groups, group_size))
    coefficients = base.view(1, taps, num_groups, group_size) + delta.unsqueeze(-1)
    output = coefficients[:, 0] * blocks
    position = torch.arange(hidden_states.shape[0], device=hidden_states.device)
    if block_size & (block_size - 1) == 0:
        position = position & (block_size - 1)
    else:
        position = position % block_size
    for tap in range(1, taps):
        shifted = F.pad(blocks[:-tap], (0, 0, 0, 0, tap, 0))
        output += coefficients[:, tap] * shifted * (position >= tap).view(-1, 1, 1)
    return output.flatten(-2)


class DFlashGroupedConv(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        taps: int,
        group_size: int,
        block_size: int,
        params_dtype: torch.dtype,
        prefix: str,
        quant_config: QuantizationConfig | None = None,
    ) -> None:
        super().__init__()
        if hidden_size % group_size:
            raise ValueError(
                f"conv_group_size={group_size} must divide hidden_size={hidden_size}."
            )
        self.block_size = block_size
        self.taps = taps
        self.group_size = group_size
        self.num_groups = hidden_size // group_size
        self.base_kernel = nn.Parameter(
            torch.empty(2, taps, hidden_size, dtype=params_dtype),
            requires_grad=False,
        )
        self.kernel_projection = ReplicatedLinear(
            hidden_size,
            2 * taps * self.num_groups,
            bias=False,
            params_dtype=params_dtype,
            quant_config=quant_config,
            prefix=maybe_prefix(prefix, "kernel_projection"),
            return_bias=False,
        )

    def _convolve(
        self, hidden_states: torch.Tensor, delta: torch.Tensor, side: int
    ) -> torch.Tensor:
        return _grouped_conv(
            hidden_states,
            delta,
            self.base_kernel[side],
            self.block_size,
            self.num_groups,
            self.group_size,
            self.taps,
        )

    def prepare(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        coefficients = self.kernel_projection(hidden_states).reshape(
            hidden_states.shape[0], 2, self.taps, self.num_groups
        )
        return self._convolve(hidden_states, coefficients[:, 0], 0), coefficients[:, 1]

    def finish(
        self, hidden_states: torch.Tensor, coefficients: torch.Tensor
    ) -> torch.Tensor:
        return self._convolve(hidden_states, coefficients, 1)


class DFlash2Qwen3DecoderLayer(DFlashQwen3DecoderLayer):
    def __init__(
        self,
        vllm_config: VllmConfig,
        *,
        config,
        layer_idx: int,
        cache_config: CacheConfig | None = None,
        quant_config: QuantizationConfig | None = None,
        prefix: str = "",
    ) -> None:
        super().__init__(
            vllm_config,
            config=config,
            layer_idx=layer_idx,
            cache_config=cache_config,
            quant_config=quant_config,
            prefix=prefix,
        )
        draft_config = config.dflash_config
        speculative_config = vllm_config.speculative_config
        assert speculative_config is not None
        conv_args = dict(
            hidden_size=config.hidden_size,
            taps=int(draft_config["conv_kernel_size"]),
            group_size=int(draft_config["conv_group_size"]),
            # Query tokens per request: the bonus token plus the mask tokens.
            block_size=1 + speculative_config.num_speculative_tokens,
            params_dtype=vllm_config.model_config.dtype,
            quant_config=quant_config,
        )
        self.attention_conv = DFlashGroupedConv(
            **conv_args, prefix=maybe_prefix(prefix, "attention_conv")
        )
        self.mlp_conv = DFlashGroupedConv(
            **conv_args, prefix=maybe_prefix(prefix, "mlp_conv")
        )

    def forward(
        self,
        positions: torch.Tensor,
        hidden_states: torch.Tensor,
        residual: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states, residual = self.input_layernorm(hidden_states, residual)

        hidden_states, coefficients = self.attention_conv.prepare(hidden_states)
        hidden_states = self.self_attn(positions=positions, hidden_states=hidden_states)
        hidden_states = self.attention_conv.finish(hidden_states, coefficients)

        hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states, coefficients = self.mlp_conv.prepare(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = self.mlp_conv.finish(hidden_states, coefficients)
        return hidden_states, residual


def _score_edges(
    predecessor_table: torch.Tensor,
    successor_table: torch.Tensor,
    candidate_ids: torch.Tensor,
    unary_logits: torch.Tensor,
    hidden: torch.Tensor,
    anchor_token_ids: torch.Tensor,
    top_k: int,
) -> torch.Tensor:
    successors = successor_table[candidate_ids]
    predecessor_ids = torch.cat(
        (
            anchor_token_ids[:, None, None].expand(-1, 1, top_k),
            candidate_ids[:, :-1],
        ),
        dim=1,
    )
    predecessors = predecessor_table[predecessor_ids]
    return unary_logits[:, :, None] + torch.einsum(
        "blpr,blcr->blpc", predecessors * hidden[:, :, None], successors
    )


@support_torch_compile
class CandidateSelector(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        vocab_size: int,
        rank: int,
        top_k: int,
        params_dtype: torch.dtype,
        prefix: str,
    ) -> None:
        super().__init__()
        self.top_k = top_k
        self.predecessor_codebook = nn.Parameter(
            torch.empty(vocab_size, rank, dtype=params_dtype), requires_grad=False
        )
        self.successor_codebook = nn.Parameter(
            torch.empty(vocab_size, rank, dtype=params_dtype), requires_grad=False
        )
        self.hidden_projection = ReplicatedLinear(
            hidden_size,
            rank,
            bias=False,
            params_dtype=params_dtype,
            quant_config=None,
            prefix=maybe_prefix(prefix, "hidden_projection"),
            return_bias=False,
        )

    def offload_codebooks(self) -> None:
        """Move read-only codebooks to pinned RAM before graph capture."""
        if getattr(self, "_cpu_codebooks", None) is not None:
            return
        names = ("predecessor_codebook", "successor_codebook")
        if (
            not current_platform.is_cuda()
            or not current_platform.is_pin_memory_available()
            or any(not getattr(self, name).is_cuda for name in names)
        ):
            raise ValueError("DFlash2 codebook offload requires pinned-memory CUDA UVA")
        if torch.cuda.is_current_stream_capturing():
            raise RuntimeError("Offload DFlash2 codebooks before CUDA graph capture")
        cpu_codebooks = {}
        views = {}
        for name in names:
            parameter = getattr(self, name)
            cpu = torch.empty_like(parameter, device="cpu", pin_memory=True)
            cpu.copy_(parameter.detach())
            cpu_codebooks[name] = cpu
            with torch.accelerator.device_index(parameter.device.index):
                views[name] = get_accelerator_view_from_cpu_tensor(cpu)
        # Keep explicit owners for the lifetime of the captured GPU pointers.
        self._cpu_codebooks = cpu_codebooks
        for name, view in views.items():
            parameter = getattr(self, name)
            parameter.data = view
            parameter._vllm_is_uva_offloaded = True
        logger.info(
            "DFlash2 selector codebooks: %.2f MiB in pinned RAM; "
            "unchanged GPU row lookup",
            sum(t.numel() * t.element_size() for t in cpu_codebooks.values()) / 2**20,
        )

    def forward(
        self,
        candidate_ids: torch.Tensor,
        unary_logits: torch.Tensor,
        hidden_states: torch.Tensor,
        anchor_token_ids: torch.Tensor,
    ) -> torch.Tensor:
        hidden = self.hidden_projection(hidden_states)
        return _score_edges(
            self.predecessor_codebook,
            self.successor_codebook,
            candidate_ids,
            unary_logits,
            hidden,
            anchor_token_ids,
            self.top_k,
        )


class DFlash2Qwen3Model(DFlashQwen3Model):
    decoder_layer_cls = DFlash2Qwen3DecoderLayer

    def __init__(
        self,
        *,
        vllm_config: VllmConfig,
        start_layer_id: int = 0,
        prefix: str = "",
    ) -> None:
        super().__init__(
            vllm_config=vllm_config,
            start_layer_id=start_layer_id,
            prefix=prefix,
        )
        draft_config = self.config.dflash_config
        self.input_embedding_scale = float(
            draft_config.get("input_embedding_scale", 1.0)
        )
        # Without its own tag the selector shares the draft head's compile cache.
        with set_model_tag("dflash2_candidate_selector"):
            self.candidate_selector = CandidateSelector(
                hidden_size=self.config.hidden_size,
                vocab_size=self.config.vocab_size,
                rank=int(draft_config["selector_rank"]),
                top_k=int(draft_config["selector_top_k"]),
                params_dtype=vllm_config.model_config.dtype,
                prefix=maybe_prefix(prefix, "candidate_selector"),
            )

    def embed_input_ids(self, input_ids: torch.Tensor) -> torch.Tensor:
        return super().embed_input_ids(input_ids) * self.input_embedding_scale


class DraftVocabHead:
    """The rows of a vocab-parallel lm_head for a subset of the vocabulary (the
    most frequent tokens, as in FR-Spec), split evenly over the TP ranks, so a
    rank projects onto len(token_ids) / tp_size rows instead of its whole shard.
    An fp8 head is applied as W8A8 with dynamic per-token activation scales."""

    def __init__(self, lm_head: nn.Module, token_ids: list[int]) -> None:
        tp = get_tp_group()
        shard = lm_head.shard_indices
        ids = [i for i in dict.fromkeys(token_ids) if 0 <= i < lm_head.org_vocab_size]
        num_ids = len(ids) // tp.world_size * tp.world_size
        if num_ids == 0:
            raise ValueError("The draft vocabulary has fewer tokens than TP ranks.")
        weight = lm_head.weight
        device = weight.device
        vocab_ids = torch.tensor(sorted(ids[:num_ids]), device=device)
        per_rank = num_ids // tp.world_size
        mine = slice(tp.rank_in_group * per_rank, (tp.rank_in_group + 1) * per_rank)
        owned = (vocab_ids >= shard.org_vocab_start_index) & (
            vocab_ids < shard.org_vocab_end_index
        )
        local_rows = vocab_ids[owned] - shard.org_vocab_start_index

        def share(rows: torch.Tensor) -> torch.Tensor:
            # Every subset row comes from the one rank whose shard holds it, so
            # a sum over zero-filled copies moves it exactly, as raw bytes.
            full = rows.new_zeros((num_ids, *rows.shape[1:]))
            full[owned] = rows[local_rows]
            raw = full.view(torch.uint8) if full.element_size() == 1 else full
            torch.distributed.all_reduce(raw, group=tp.device_group)
            return full[mine].contiguous()

        # PWAL can transpose storage while preserving checkpoint loader axes.
        input_dim, output_dim = getattr(
            lm_head,
            "processed_weight_dims",
            (getattr(weight, "input_dim", 1), getattr(weight, "output_dim", 0)),
        )
        if (input_dim, output_dim) not in ((1, 0), (0, 1)):
            raise ValueError("Unsupported draft lm_head weight layout")
        rows = weight.t() if output_dim == 1 else weight
        if rows.shape != (lm_head.num_embeddings_per_partition, lm_head.embedding_dim):
            raise ValueError(
                f"Draft lm_head shape {tuple(weight.shape)} does not match "
                f"its processed layout {(input_dim, output_dim)}"
            )
        self.weight_scale: torch.Tensor | None = None
        if weight.dtype == current_platform.fp8_dtype():
            # cutlass_scaled_mm takes the weight column-major, as [hidden, vocab].
            self.weight = share(rows).t()
            scale = lm_head.weight_scale.float()
            self.weight_scale = (
                share(scale.reshape(rows.shape[0], 1)) if scale.numel() > 1 else scale
            )
        else:
            self.weight = share(rows)
        self.vocab_ids = vocab_ids[mine]
        self.tp_size = tp.world_size

    def __call__(self, hidden_states: torch.Tensor) -> torch.Tensor:
        if self.weight_scale is None:
            return F.linear(hidden_states, self.weight)
        x, x_scale = ops.scaled_fp8_quant(hidden_states, use_per_token_if_dynamic=True)
        return ops.cutlass_scaled_mm(
            x, self.weight, x_scale, self.weight_scale, hidden_states.dtype
        )


class DFlash2Qwen3ForCausalLM(DFlashQwen3ForCausalLM):
    model_cls = DFlash2Qwen3Model

    def __init__(self, *, vllm_config: VllmConfig, prefix: str = "") -> None:
        super().__init__(vllm_config=vllm_config, prefix=prefix)
        draft_config = self.config.dflash_config
        softcap = float(draft_config.get("final_logit_softcapping") or 0.0)
        self.candidate_logits_processor = LogitsProcessor(
            vllm_config.model_config.get_vocab_size(),
            scale=float(draft_config.get("output_multiplier", 1.0)),
            soft_cap=softcap if softcap > 0 else None,
        )
        self.draft_vocab_head: DraftVocabHead | None = None

    def offload_selector_codebooks(self) -> None:
        self.model.candidate_selector.offload_codebooks()

    def restrict_draft_vocab(self, token_ids: list[int]) -> None:
        """Draws candidates from these tokens only; call on every TP rank once
        the lm_head is loaded (and shared with the target)."""
        head = DraftVocabHead(self.lm_head, token_ids)
        self.draft_vocab_head = head
        logger.info_once(
            "DFlash2 drafts from %d tokens (%d per TP rank)",
            head.vocab_ids.numel() * head.tp_size,
            head.vocab_ids.numel(),
        )

    def compute_candidates(
        self, hidden_states: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        top_k = self.model.candidate_selector.top_k
        head = self.draft_vocab_head
        if head is not None:
            return self.candidate_logits_processor.get_subset_top_k_tokens(
                head(hidden_states), head.vocab_ids, top_k, head.tp_size
            )
        return self.candidate_logits_processor.get_top_k_tokens(
            self.lm_head, hidden_states, top_k
        )


EntryClass = DFlash2Qwen3ForCausalLM


In [ ]:
import os
os.environ.update({
    "FASTSAFETENSORS_ODIRECT": "0",
    "VLLM_HOST_ALLREDUCE": "1",
    "VLLM_HOST_ALLREDUCE": "1",
    "VLLM_HOST_ALLREDUCE_MODE": "60",
    "VLLM_HOST_ALLREDUCE_PREFETCH_BLOCKS": "16",
    "VLLM_HOST_ALLREDUCE_PREFETCH_MAX_MB": "36",
    "VLLM_GDN_DEFERRED_STATE_COMMIT": "stream",
    "VLLM_GDN_DEFERRED_FULL_GRAPH": "1",
    "VLLM_GDN_DEFERRED_LATE": "1",
    "VLLM_FLASHINFER_FA2_SPEC_DECODE": "1",
    "VLLM_EARLY_SPEC_SEQ_LENS": "1",
    "VLLM_BATCH_SHARD_TOPK": "32",
    "VLLM_GDN_SINGLE_STATE_BLOCK": "1",
    "VLLM_MARLIN_PAD_M": "5120x4352:32",
    "VLLM_DRAFT_VOCAB_FILE": "/kaggle/working/fr32k.json",
    "VLLM_KV_CACHE_GROUP_SIZE": "8",
    "VLLM_SM89_FP8_TILES": "128:4096x5120=t32x128x128_w32x64x64_s3_f,128:62080x5120=t80x64x128_w80x32x64_s5_f,128:8704x5120=t16x128x64_w16x64x64_s5_f,128:5120x4352=t80x64x128_w80x32x64_s5_f,128:3584x5120=t80x64x128_w80x32x64_s3_f,128:5120x1536=t80x64x128_w80x32x64_s5_f,64:4096x5120=t64x64x128_w32x64x64_s3_s,64:5120x1536=t64x64x128_w32x64x64_s3_s,64:3584x5120=t64x64x128_w32x64x64_s3_s,64:5120x4352=t64x64x128_w32x64x64_s3_s,64:8704x5120=t32x64x128_w16x64x64_s6_f,16:4096x5120=t32x128x128_w32x64x64_s3_f,16:5120x4352=t32x128x128_w32x64x64_s3_f,81-96:62080x5120=t96x64x128_w96x32x64_s4_f,81-96:3584x5120=t96x64x128_w96x32x64_s3_f,81-96:5120x1536=t96x64x128_w96x32x64_s2_f",
    # "VLLM_SM89_FP8_TILES": "128:4096x5120=t32x128x128_w32x64x64_s3_f,128:62080x5120=t80x64x128_w80x32x64_s5_f,128:8704x5120=t16x128x64_w16x64x64_s5_f,128:5120x4352=t80x64x128_w80x32x64_s5_f,128:3584x5120=t80x64x128_w80x32x64_s3_f,128:5120x1536=t80x64x128_w80x32x64_s5_f,64:4096x5120=t64x64x128_w32x64x64_s3_s,64:5120x1536=t64x64x128_w32x64x64_s3_s,64:3584x5120=t64x64x128_w32x64x64_s3_s,64:5120x4352=t64x64x128_w32x64x64_s3_s,64:8704x5120=t32x64x128_w16x64x64_s6_f,16:4096x5120=t32x128x128_w32x64x64_s3_f,16:5120x4352=t32x128x128_w32x64x64_s3_f",
})

import os

cudnn_root = "/usr/local/lib/python3.13/dist-packages/nvidia/cudnn"
os.environ["CUDNN_HOME"] = cudnn_root
os.environ["CUDNN_PATH"] = cudnn_root
os.environ["LD_LIBRARY_PATH"] = (
    cudnn_root + "/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
)


qwen38_root = f"/kaggle/working/arc_qwen38_x"
# !ARC_TASK_KEY=0934a4d8,36a08778,981571dc,aa4ec2a5 \
!ARC_RUN_STATUS_PATH=/kaggle/working/arc_run_status.json \
python /kaggle/working/qwen38_pipeline.py \
  --qwen38-recipe vision-repl-audit \
  --model-qwen38 /kaggle/input/models/nguyennguyen599/qwen3-8/transformers/unsloth-qwen3.8-27b-nvfp4/1 \
  --qwen38-draft-model /kaggle/input/models/nguyennguyen599/qwen3-8/transformers/yourhighnessla-qwen3.8-27b-dflash2-nvfp4/1 \
  --output-root {qwen38_root} \
  --final-submission /kaggle/working/submission.json \
  --end-time {global_end_time} \
  --qwen38-max-seconds 0 \
  --finalization-reserve-seconds 360 \
  --tensor-parallel-size 4 \
  --qwen38-port 18083 \
  --temp-root /tmp/arc_vision \
  --context 128000 \
  --concurrency 24 \
  --gpu-memory-utilization 0.94 \
  --max-num-batched-tokens 2048 \
  --draft-tokens 4 \
  --draft-quantization auto \
  --load-format fastsafetensors \
  --fastsafetensors-max-gpu-shard-gib 8 \
  --vision-max-pixels 2097152 \
  --loop-gzip-action stop \
  --loop-gzip-window-tokens 1000 \
  --loop-gzip-ratio 8 \
  --loop-gzip-check-tokens 1000 \
  --loop-gzip-confirmations 3 \
  --task-token-budget 70000 \
  --tir-repl-budget 70000 \
  --tir-repl-trajectories 1 \
  --queue-order small-area \
  --vllm-arg=--enable-batch-sharded-sampling \
  '--vllm-arg=--compilation-config.cudagraph_capture_sizes=[1,2,4,5,8,10,15,16,20,24,25,30,32,35,40,45,48,50,55,56,60,64,65,70,72,75,80,88,96,104,112,120,128,136,144,152,160]' \
  --min-request-tokens 30000
# 65536
  # --min-request-tokens 60000 \
# --input-mode text-image

In [ ]:
!ps aux | grep vllm

In [ ]:
!kill 5710

In [ ]:
# assert 1==0

In [ ]:
# !cat /kaggle/working/arc_4b2qwen38/qwen38/server/server.log

In [ ]:
!du -h -ah --max-depth=1 /kaggle/working/ | sort -rh

In [ ]:
# !tar --use-compress-program='pigz -9' -cf /kaggle/working/log.tar.gz -C /kaggle/inference_outputs .
# !tar --use-compress-program='pigz -9' -cf /kaggle/working/log_cs.tar.gz -C /kaggle/working/arc_4b2qwen38 .
# !tar --use-compress-program='pigz -9' -cf /kaggle/working/log_cs_1.tar.gz -C /kaggle/working/arc_qwen38_x .

!mkdir /kaggle/tmp
!zip -r /kaggle/tmp/log.tar.gz /kaggle/inference_outputs
!zip -r /kaggle/tmp/log_cs.tar.gz /kaggle/working/arc_4b2qwen38
!zip -r /kaggle/tmp/log_cs_1.tar.gz /kaggle/working/arc_qwen38_x

!cp /kaggle/working/submission.json /kaggle/tmp/submission.json
!rm -r /kaggle/working/*

!cp /kaggle/tmp/submission.json /kaggle/working/submission.json
!cp /kaggle/tmp/log.tar.gz /kaggle/working/log.tar.gz
!cp /kaggle/tmp/log_cs.tar.gz /kaggle/working/log_cs.tar.gz
!cp /kaggle/tmp/log_cs_1.tar.gz /kaggle/working/log_cs_1.tar.gz

# !rm -r /kaggle/working/arc_4b2qwen38
# !rm -r /kaggle/working/arc_qwen38_x